In [2]:
import pandas as pd
import random
from datetime import timedelta, date
from faker import Faker

# Инициализация генератора фейковых данных (для имен и описаний)
fake = Faker('ru_RU')

def generate_random_date(start_date, end_date):
    """Генерирует случайную дату между start_date и end_date."""
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_number_of_days = random.randrange(days_between_dates)
    return start_date + timedelta(days=random_number_of_days)

def generate_dataset(num_rows=10000):
    print(f"Генерация датасета из {num_rows} строк...")
    
    # Определяем диапазон года для платежей (например, 2023 год)
    year_start = date(2025, 1, 1)
    year_end = date(2025, 12, 31)
    
    data = {
        'payment_date': [],
        'contract_start_date': [],
        'contract_end_date': [],
        'amount': [],
        'currency': [],
        'transaction_type': [],
        'counterparty': [],
        'description': []
    }

    for _ in range(num_rows):
        # 1. Генерируем дату платежа (якорь)
        p_date = generate_random_date(year_start, year_end)
        
        # 2. Генерируем дату начала договора (обязательно ДО платежа)
        # Допустим, договор мог начаться от 1 дня до 2 лет назад
        min_start = p_date - timedelta(days=730) 
        c_start = generate_random_date(min_start, p_date - timedelta(days=1))
        
        # 3. Генерируем дату завершения договора (обязательно ПОСЛЕ платежа)
        # Допустим, договор заканчивается от 1 дня до 2 лет вперед
        max_end = p_date + timedelta(days=730)
        c_end = generate_random_date(p_date + timedelta(days=1), max_end)
        
        # Заполняем обязательные столбцы
        data['payment_date'].append(p_date)
        data['contract_start_date'].append(c_start)
        data['contract_end_date'].append(c_end)
        
        # Генерируем дополнительные реалистичные данные
        amount = round(random.uniform(100.0, 500000.0), 2)
        # Делаем некоторые суммы отрицательными (списание), некоторые положительными (зачисление)
        if random.random() > 0.5:
            amount = -amount
            
        data['amount'].append(amount)
        data['currency'].append('RUB')
        data['transaction_type'].append(random.choice(['Перевод', 'Оплата по счету', 'Снятие наличных', 'Зачисление зарплаты', 'Комиссия']))
        data['counterparty'].append(fake.company() if random.random() > 0.3 else fake.name())
        data['description'].append(fake.sentence(nb_words=6))

    # Создаем DataFrame
    df = pd.DataFrame(data)
    
    # Сортируем по дате платежа для реалистичности выписки
    df = df.sort_values(by='payment_date').reset_index(drop=True)
    
    return df

def save_to_excel(df, filename='C:/Users/fedor/Downloads/bank_statement_2025.xlsx'):
    try:
        # Сохраняем в Excel
        df.to_excel(filename, index=False)
        print(f"Успешно! Файл сохранен как: {filename}")
        print(f"Количество строк: {len(df)}")
        print(f"Столбцы: {list(df.columns)}")
    except Exception as e:
        print(f"Ошибка при сохранении файла: {e}")

if __name__ == "__main__":
    # Генерируем данные
    dataframe = generate_dataset(10000)
    
    # Сохраняем в файл
    save_to_excel(dataframe)


Генерация датасета из 10000 строк...
Успешно! Файл сохранен как: C:/Users/fedor/Downloads/bank_statement_2025.xlsx
Количество строк: 10000
Столбцы: ['payment_date', 'contract_start_date', 'contract_end_date', 'amount', 'currency', 'transaction_type', 'counterparty', 'description']
